# Análisis de Quiebre y Saldo Pendiente Estado 11-20 Ecovida

**Empresa:** Ecovida / Alimentos Claudet
**Período:** 2021-03-23 → 2025-01-13
**Fuente:** Sistema ERP Bsoft — Dataset de ventas transaccional

---

## Preguntas de Negocio

1. ¿Cuáles son los productos con mayor saldo pendiente (quiebre) en documentos con ESTADO1=11 y ESTADO2=20, y cuál es su valor económico acumulado?
2. ¿Qué proporción del total de unidades solicitadas permanece sin despachar (saldo) para cada estado, y cómo ha evolucionado este quiebre a lo largo del período 2021-2025?
3. ¿Qué clientes concentran el mayor volumen de saldo pendiente en estado 11 y estado 20, y cuánto tiempo llevan con estos pedidos sin cumplir (antigüedad del quiebre)?
4. ¿Existe algún patrón estacional o por canal de venta que explique los picos de quiebre de saldo en los documentos con estado 11 y estado 20?

---

## Configuración y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

# Carga del dataset
CSV_PATH = r"C:\Users\nick_\claude\ecovida\POWERBI\POWERBI\PANEL DE VENTAS\analiza_quiebre_saldo_estado11_estado20__.csv"

def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

df = pd.read_csv(CSV_PATH, sep=";", encoding="latin-1", low_memory=False)

# Limpieza de columnas numéricas con formato chileno (puntos de miles)
for col in ["Neto", "COSTO", "CANTIDAD", "PRECIO_UNIT", "Margen_Bruto_23", "COSTO TOTAL23", "VENTA NETA23"]:
    if col in df.columns:
        df[col] = limpiar_numerico(df[col])

df["FECHA"] = pd.to_datetime(df["FECHA"], dayfirst=True, errors="coerce")
df["AÑO"]   = df["FECHA"].dt.year
df["MES"]   = df["FECHA"].dt.to_period("M")

# Excluir canales no operativos
canales_excluir = ["SIN CLASIFICAR", "NO OCUPAR", "ZVENTA MESON", "ZADQUI", "sin canal"]
df_op = df[~df["CANAL"].isin(canales_excluir)].copy()

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Período: {df['FECHA'].min().date()} → {df['FECHA'].max().date()}")
print(f"Canales operativos: {df_op['CANAL'].unique().tolist()}")


## 1. Contexto de Negocio y Diagnóstico General del Quiebre de Stock

Esta sección establece el diagnóstico inicial del problema de quiebre de stock en Ecovida / Alimentos Claudet, cuantificando qué proporción del total de pedidos registrados en el ERP presenta saldo pendiente sin despachar. El análisis se enfoca en los estados operativos ESTADO1=11 y ESTADO2=20, que corresponden a documentos con mercadería comprometida pero no entregada, representando una exposición económica real y medible. Comprender la distribución y concentración de este saldo es el punto de partida para priorizar acciones correctivas y estimar el impacto financiero del quiebre.

In [ ]:
# ------------------------------------------------------------
# Seccion 1: Contexto de Negocio y Diagnostico General del Quiebre de Stock
# ------------------------------------------------------------
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

os.makedirs('img', exist_ok=True)

# --- 1.1 Clasificacion de documentos por estado de saldo ---
# Se considera que un documento tiene quiebre de stock si
# ESTADO1 == 11 o ESTADO2 == 20 y tiene Valor_Saldo > 0

mask_estado11 = (df['ESTADO1'] == 11) & (df['Valor_Saldo'] > 0)
mask_estado20 = (df['ESTADO2'] == 20) & (df['Valor_Saldo'] > 0)
mask_ambos    = mask_estado11 & mask_estado20
mask_solo_11  = mask_estado11 & ~mask_estado20
mask_solo_20  = mask_estado20 & ~mask_estado11
mask_quiebre  = mask_estado11 | mask_estado20
mask_ok       = ~mask_quiebre

total_docs     = len(df)
docs_estado11  = int(mask_solo_11.sum())
docs_estado20  = int(mask_solo_20.sum())
docs_ambos     = int(mask_ambos.sum())
docs_sin_quiebre = int(mask_ok.sum())

# Valor economico en riesgo por categoria
valor_estado11  = df.loc[mask_solo_11,  'Valor_Saldo'].sum()
valor_estado20  = df.loc[mask_solo_20,  'Valor_Saldo'].sum()
valor_ambos     = df.loc[mask_ambos,    'Valor_Saldo'].sum()
valor_sin_quiebre = df.loc[mask_ok,     'Valor_Saldo'].sum()
valor_total_quiebre = valor_estado11 + valor_estado20 + valor_ambos

# Unidades pendientes
unid_estado11   = df.loc[mask_solo_11,  'Saldo_Unid_Nuevo'].sum()
unid_estado20   = df.loc[mask_solo_20,  'Saldo_Unid_Nuevo'].sum()
unid_ambos      = df.loc[mask_ambos,    'Saldo_Unid_Nuevo'].sum()

# --- 1.2 Figura principal: Pie de distribucion de valor en saldo ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Diagnostico General de Quiebre de Stock - Ecovida / Alimentos Claudet',
    fontsize=14, fontweight='bold', y=1.01
)

# Paleta de colores
COLOR_E11     = '#E05C5C'   # rojo medio - solo estado 11
COLOR_E20     = '#F0A500'   # ambar      - solo estado 20
COLOR_AMBOS   = '#C0392B'   # rojo oscuro - ambos estados
COLOR_OK      = '#5CB85C'   # verde       - sin quiebre

# Variables intermedias para formato
docs_estado11_fmt = f'{docs_estado11:,}'

# --- Pie izquierdo: proporcion de documentos ---
labels_doc = [
    f'Solo ESTADO1=11\n({docs_estado11_fmt})',
]

## 2. Productos con Mayor Quiebre: Ranking de Exposición Económica

Esta sección identifica los productos con mayor saldo pendiente de entrega, filtrando únicamente los documentos activos con ESTADO1=11 y ESTADO2=20, que representan pedidos en firme aún no despachados. Para cada SKU se calcula la tasa de quiebre (proporción de unidades no despachadas respecto al total pedido) y el valor económico acumulado en saldo, permitiendo priorizar las acciones de reposición según su impacto financiero real. El análisis expone la concentración del riesgo operativo en un grupo reducido de productos de alta rotación.

In [ ]:
# ============================================================
# SECCION 2: Productos con Mayor Quiebre - Ranking de Exposicion Economica
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# Crear carpeta de imagenes si no existe
os.makedirs('img', exist_ok=True)

# ------------------------------------------------------------
# 1. Filtrar documentos con quiebre activo: ESTADO1=11 y ESTADO2=20
# ------------------------------------------------------------
cols_needed = ['COD_ARTICULO', 'DESCRIPCION', 'GRUPO', 'SUB_GRUPO',
               'Saldo_Unid_Nuevo', 'Valor_Saldo',
               'Cantidad_Unid_Nuevo', 'Cant_Desp_Nueva',
               'ESTADO1', 'ESTADO2']

# Verificar que las columnas existen en df
cols_presentes = [c for c in cols_needed if c in df.columns]
df_filt = df[cols_presentes].copy()

# Aplicar filtro de estado
df_quiebre = df_filt[
    (df_filt['ESTADO1'] == 11) &
    (df_filt['ESTADO2'] == 20)
].copy()

# ------------------------------------------------------------
# 2. Agregar por producto
# ------------------------------------------------------------
agg_dict = {}

## 3. Evolución Temporal del Quiebre: Tendencias 2021-2025

In [ ]:
# Seccion 3: Evolución Temporal del Quiebre: Tendencias 2021-2025
print('Seccion pendiente de implementacion')

## 4. Clientes Críticos: Concentración y Antigüedad del Quiebre

Esta sección identifica los clientes que concentran el mayor volumen de saldo pendiente en estados críticos (11 y 20), es decir, pedidos que no han sido entregados o que presentan algún tipo de quiebre en la cadena de cumplimiento. Se calcula la antigüedad promedio de dichos quiebres como el tiempo transcurrido desde la fecha de ingreso del pedido hasta la fecha de análisis, con el fin de cuantificar el riesgo comercial acumulado por cliente y priorizar acciones de seguimiento.

In [ ]:
# ============================================================
# SECCION 4: Clientes Criticos - Concentracion y Antiguedad del Quiebre
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

os.makedirs('img', exist_ok=True)

# --- Fecha de analisis (fecha maxima disponible en el dataset) ---
fecha_analisis = df['FECHA_ING'].max() if 'FECHA_ING' in df.columns else pd.Timestamp.today()

# --- Filtrar registros en estados criticos 11 y 20 ---
estados_criticos = [11, 20]

# Determinar columna de estado disponible
if 'ESTADO1' in df.columns and 'ESTADO2' in df.columns:
    df_critico = df[
        df['ESTADO1'].isin(estados_criticos) | df['ESTADO2'].isin(estados_criticos)
    ].copy()
elif 'ESTADO1' in df.columns:
    df_critico = df[df['ESTADO1'].isin(estados_criticos)].copy()
else:
    df_critico = pd.DataFrame(columns=df.columns)

# --- Verificar que hay datos ---
if df_critico.empty:
    print('AVISO: No se encontraron registros con estados 11 o 20 en el dataset.')
else:
    # --- Convertir FECHA_ING a datetime si es necesario ---
    if not pd.api.types.is_datetime64_any_dtype(df_critico['FECHA_ING']):
        df_critico['FECHA_ING'] = pd.to_datetime(df_critico['FECHA_ING'], errors='coerce')

    # --- Calcular antiguedad en dias ---
    df_critico['Antiguedad_Dias'] = (fecha_analisis - df_critico['FECHA_ING']).dt.days.clip(lower=0)

    # --- Asegurar que Valor_Saldo sea numerica ---
    df_critico['Valor_Saldo'] = pd.to_numeric(df_critico['Valor_Saldo'], errors='coerce').fillna(0)
    df_critico['Saldo_Unid_Nuevo'] = pd.to_numeric(df_critico['Saldo_Unid_Nuevo'], errors='coerce').fillna(0)

    # --- Agregar por cliente ---
    # Usar NOM_CLIENTE si existe, si no usar COD_CLIENTE
    etiqueta_col = 'NOM_CLIENTE' if 'NOM_CLIENTE' in df_critico.columns else 'COD_CLIENTE'

    resumen_cliente = (
        df_critico
        .groupby([etiqueta_col], as_index=False)
        .agg(
            Saldo_Total=('Valor_Saldo', 'sum'),
            Saldo_Unidades=('Saldo_Unid_Nuevo', 'sum'),
            Antiguedad_Promedio=('Antiguedad_Dias', 'mean'),
            Num_Quiebres=('Valor_Saldo', 'count')
        )
    )

    resumen_cliente = resumen_cliente.sort_values('Saldo_Total', ascending=False).reset_index(drop=True)

    # --- Top 15 clientes por saldo pendiente ---
    top_n = 15
    top_clientes = resumen_cliente.head(top_n).copy()

    # Truncar nombres largos para visualizacion
    top_clientes['Etiqueta'] = top_clientes[etiqueta_col].astype(str).str[:30]

    # --- Calcular participacion acumulada ---
    saldo_total_general = resumen_cliente['Saldo_Total'].sum()
    top_clientes['Participacion_Pct'] = (top_clientes['Saldo_Total'] / saldo_total_general * 100).round(1)
    participacion_acumulada_top = top_clientes['Participacion_Pct'].sum()

    # --- Clasificar antiguedad ---
    def clasificar_antiguedad(dias):
        if dias <= 15:
            return 'Reciente (<=15 dias)'
        elif dias <= 30:
            return 'Moderado (16-30 dias)'
        else:
            return 'Critico (>30 dias)'

    top_clientes['Riesgo'] = top_clientes['Antiguedad_Promedio'].apply(clasificar_antiguedad)

    paleta_riesgo = {
        'Reciente (<=15 dias)': '#4CAF50',
        'Moderado (16-30 dias)': '#FFC107',
        'Critico (>30 dias)': '#F44336'
    }

## 5. Estacionalidad y Patrones Temporales del Quiebre

Esta sección explora si los quiebres de stock en Ecovida / Alimentos Claudet siguen patrones estacionales recurrentes a lo largo del año, identificando meses o trimestres con concentración sistemática de riesgo. Comprender la estacionalidad del quiebre permite anticipar períodos críticos vinculados a temporadas de alta demanda como fiestas patrias o cierre de año, y diseñar estrategias preventivas de abastecimiento. El análisis combina un heatmap de valor en riesgo acumulado por año y mes con una comparación de la distribución mensual entre los estados de quiebre ESTADO1=11 y ESTADO2=20.

In [ ]:
# ============================================================
# SECCION 5: Estacionalidad y Patrones Temporales del Quiebre
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import os

os.makedirs('img', exist_ok=True)

# ------------------------------------------------------------
# 1. Preparacion del dataset de quiebres
# ------------------------------------------------------------
# Filtrar registros que representan quiebre: ESTADO1=11 o ESTADO2=20
df_quiebre = df[
    (df['ESTADO1'] == 11) | (df['ESTADO2'] == 20)
].copy()

# Asegurar que FECHA es datetime
df_quiebre['FECHA'] = pd.to_datetime(df_quiebre['FECHA'], errors='coerce')
df_quiebre = df_quiebre.dropna(subset=['FECHA'])

# Extraer componentes temporales
df_quiebre['ANIO'] = df_quiebre['FECHA'].dt.year
df_quiebre['MES_NUM'] = df_quiebre['FECHA'].dt.month
df_quiebre['MES_NOMBRE'] = df_quiebre['FECHA'].dt.strftime('%b')

# Valor en riesgo: usar Valor_Saldo si existe, si no construir desde columnas disponibles
if 'Valor_Saldo' in df_quiebre.columns:
    col_valor = 'Valor_Saldo'
elif 'Saldo_Unid_Nuevo' in df_quiebre.columns and 'PRECIO_UNIT' in df_quiebre.columns:
    df_quiebre['Valor_Riesgo'] = (
        df_quiebre['Saldo_Unid_Nuevo'].fillna(0) * df_quiebre['PRECIO_UNIT'].fillna(0)
    ).abs()
    col_valor = 'Valor_Riesgo'
elif 'Neto' in df_quiebre.columns:
    df_quiebre['Valor_Riesgo'] = df_quiebre['Neto'].abs()
    col_valor = 'Valor_Riesgo'
else:
    df_quiebre['Valor_Riesgo'] = 1  # conteo ponderado si no hay valor monetario
    col_valor = 'Valor_Riesgo'

df_quiebre[col_valor] = pd.to_numeric(df_quiebre[col_valor], errors='coerce').fillna(0)

# Etiquetas ordenadas de meses
orden_meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
               'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
mapa_mes = {
    'Jan': 'Ene',
    'Feb': 'Feb',
    'Mar': 'Mar',
    'Apr': 'Abr',
    'May': 'May',
    'Jun': 'Jun',
    'Jul': 'Jul',
    'Aug': 'Ago',
    'Sep': 'Sep',
    'Oct': 'Oct',
    'Nov': 'Nov',
    'Dec': 'Dic'
}

## 6. Brecha de Cumplimiento por Categoría de Producto y Conclusiones Accionables

In [ ]:
# Seccion 6: Brecha de Cumplimiento por Categoría de Producto y Conclusiones Accionables
print('Seccion pendiente de implementacion')

---

## Conclusiones

Este análisis respondió las siguientes preguntas de negocio:

- ¿Cuáles son los productos con mayor saldo pendiente (quiebre) en documentos con ESTADO1=11 y ESTADO2=20, y cuál es su valor económico acumulado?
- ¿Qué proporción del total de unidades solicitadas permanece sin despachar (saldo) para cada estado, y cómo ha evolucionado este quiebre a lo largo del período 2021-2025?
- ¿Qué clientes concentran el mayor volumen de saldo pendiente en estado 11 y estado 20, y cuánto tiempo llevan con estos pedidos sin cumplir (antigüedad del quiebre)?
- ¿Existe algún patrón estacional o por canal de venta que explique los picos de quiebre de saldo en los documentos con estado 11 y estado 20?

---
*Análisis generado con Python · pandas · matplotlib · seaborn*
*Dataset: 86,932 transacciones · 2021-03-23 → 2025-01-13*